# 05 - Baselines (two tiers, three seeds)

**Runtime -> Run all.** Every run saves per-domain predictions and a manifest
line; completed runs are detected and skipped, so re-running after a dead
runtime resumes rather than repeats.

Two evaluation tiers, because certificate features only have honest semantics
where every domain was actually probed:

* **Tier A - full corpus (1.78M), lexical features only.** In the full matrix,
  98.3% of rows were never probed, so `has_certificate=False` would mostly
  mean "not probed" rather than "no TLS". Certificate features are therefore
  excluded here.
* **Tier B - probe universe (~50k), lexical + certificate.** Every row was
  probed; absence of a certificate is a real observation. The fusion claim is
  evaluated in this tier.

Models: logistic regression and random forest, on numeric features with
missing values filled sentinel (-1). These are yardsticks, not contributions -
the categorical and missing-value handling is deliberately naive so that
XGBoost's gains in notebook 06 are attributable.

Three seeds per configuration; headline numbers are reported as mean +/- std.
Cross-split comparisons use prevalence-free metrics (ROC-AUC, FPR@95%TPR)
because the random and family-disjoint test sets have different base rates
(46.9% vs 65.5% malicious).

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard scikit-learn

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from src.evaluate import splits, metrics, predictions
from src.utils import manifest as mf

FEATURES = Path(P['data']['features'])
X_all = pd.read_parquet(FEATURES/'fused_v1.parquet')
probe = pd.read_parquet(f"{P['data']['interim']}/probe_universe.parquet")
probe_set = set(probe['domain'])
print('full matrix :', X_all.shape)
print('probe subset:', X_all['domain'].isin(probe_set).sum())

## Feature sets per tier

In [ ]:
LEXICAL = ['length','core_length','n_labels','shannon_entropy','vowel_ratio',
           'digit_ratio','hyphen_count','max_consec_consonants','bigram_score',
           'trigram_score','unique_char_ratio','is_idn','has_digit','starts_with_digit']

CERT_NUM = ['has_certificate','is_free_ca','validity_days','days_until_expiry',
            'cert_age_days','is_expired','is_not_yet_valid','is_self_signed',
            'san_count','wildcard_san','cn_in_san','key_bits','short_validity',
            'very_fresh_cert','cn_san_mismatch','weak_key']

def matrix(df, cols):
    Xm = df[cols].copy()
    for c in Xm.columns:                      # bools and objects -> numeric
        if Xm[c].dtype == object or Xm[c].dtype == bool:
            Xm[c] = Xm[c].astype(float, errors='ignore')
    Xm = Xm.apply(pd.to_numeric, errors='coerce').fillna(-1.0)
    return Xm.values.astype(np.float32), df['label'].values, df['domain'].values

TIERS = {
    'tierA_lexical': (X_all, LEXICAL),
    'tierB_lexcert': (X_all[X_all['domain'].isin(probe_set)].copy(), LEXICAL + CERT_NUM),
}
for name, (d, cols) in TIERS.items():
    print(f'{name:16s} rows={len(d):>9,}  features={len(cols)}')

## Run grid

model x split x seed x tier. A run whose predictions file already exists is
skipped - that is the resume mechanism.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def build_model(name, seed):
    if name == 'logreg':
        return ('scaled', LogisticRegression(C=1.0, max_iter=2000,
                    class_weight='balanced', random_state=seed))
    if name == 'rf':
        return ('raw', RandomForestClassifier(n_estimators=300, n_jobs=-1,
                    class_weight='balanced_subsample', random_state=seed))
    raise ValueError(name)

SPLITS = ['random_v1', 'family_disjoint_v1']
SEEDS  = [42, 43, 44]
MODELS = ['logreg', 'rf']
PRED_DIR = Path(P['artifacts']['predictions'])

results = []
for tier, (data, cols) in TIERS.items():
    for split_name in SPLITS:
        sp = splits.load_split(P['data']['splits'], split_name)
        tr = data[data['domain'].isin(sp['domains']['train'])]
        te = data[data['domain'].isin(sp['domains']['test'])]
        Xtr, ytr, _   = matrix(tr, cols)
        Xte, yte, dte = matrix(te, cols)
        for model_name in MODELS:
            for seed in SEEDS:
                run_id = f'{model_name}_{tier}_{split_name}_s{seed}'
                if (PRED_DIR/f'{run_id}.parquet').exists():
                    print('SKIP (done)   ', run_id)
                    continue
                kind, model = build_model(model_name, seed)
                if kind == 'scaled':
                    sc = StandardScaler().fit(Xtr)
                    model.fit(sc.transform(Xtr), ytr)
                    scores = model.predict_proba(sc.transform(Xte))[:,1]
                else:
                    model.fit(Xtr, ytr)
                    scores = model.predict_proba(Xte)[:,1]
                m = metrics.evaluate(yte, scores)
                predictions.save(run_id, PRED_DIR, dte, yte, scores)
                mf.record(P['manifest'], run_id, f'baseline_{tier}',
                          {'model': model_name, 'tier': tier, 'features': cols},
                          split_name, sp['split_file'], m, seed, repo_root=REPO)
                results.append({'run_id': run_id, **m})
                print(f'DONE {run_id:45s} roc={m["roc_auc"]:.4f} '
                      f'fpr@95={m["fpr_at_95_tpr"]:.4f}')
print('grid complete')

## Summary - mean +/- std over seeds

In [ ]:
man = mf.load_manifest(P['manifest'])
base = man[man['run_family'].str.startswith('baseline')].copy()
base['model'] = base['run_id'].str.split('_').str[0]
base['tier']  = base['run_id'].str.extract(r'(tier[AB]_\w+?)_')[0]

agg = (base.groupby(['tier','model','split_name'])
       [['metrics.roc_auc','metrics.pr_auc','metrics.fpr_at_95_tpr','metrics.mcc']]
       .agg(['mean','std']).round(4))
display(agg)

out = Path(P['results']['tables'])/'table_baselines.csv'
agg.to_csv(out)
print('wrote', out)

In [ ]:
# The two numbers that frame everything downstream:
# 1. the random -> family-disjoint generalisation drop (prevalence-free metric)
# 2. tier B vs tier A on the same split: what certificates add over lexical
piv = (base.groupby(['tier','model','split_name'])['metrics.roc_auc']
       .mean().unstack('split_name').round(4))
piv['generalisation_drop'] = (piv['random_v1'] - piv['family_disjoint_v1']).round(4)
display(piv)

---

**Next:** `06_xgboost` (same two tiers, categorical features included, three
seeds), then the CNN-BiLSTM. The baseline table above is the yardstick every
later model must beat - and the generalisation-drop column is the honesty
metric that goes in the paper regardless of how it looks.